<a href="https://colab.research.google.com/github/darjiavani/college-website/blob/main/GAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

In [ ]:
data = pd.read_csv("/content/energy_architecture_data.csv")
data.head()

,floor_area,number_of_floors,window_to_wall_ratio,wall_thickness,avg_temperature,avg_humidity,solar_radiation,thermal_resistance,estimated_energy_usage,daylight_factor,thermal_load
0,281.4,2,0.49,0.37,17.1,61,415,0.10,61362.6,2.03,-807.9
1,186.7,2,0.30,0.45,-0.4,24,116,5.18,11.8,0.35,-1835.8
2,198.4,2,0.30,0.42,19.3,43,138,0.08,55437.2,0.41,-137.8
3,154.1,2,0.15,0.42,-7.0,72,484,6.22,173.5,0.73,-1572.7
4,123.8,2,0.30,0.37,23.0,67,937,3.24,1051.8,2.81,251.1


In [6]:
input_dim = 100
output_dim = data.shape[1]

def build_generator():
    model = tf.keras.Sequential([
        layers.Dense(128, activation="relu", input_dim=input_dim),
        layers.Dense(256, activation="relu"),
        layers.Dense(512, activation="relu"),
        layers.Dense(output_dim, activation="tanh")  # Output layer
    ])
    return model




In [8]:
def build_discriminator():
    model = tf.keras.Sequential([
        layers.Dense(512, activation="relu", input_shape=(output_dim,)),
        layers.Dense(256, activation="relu"),
        layers.Dense(128, activation="relu"),
        layers.Dense(1, activation="sigmoid")  # Binary classification
    ])
    return model






In [10]:
generator = build_generator()
discriminator = build_discriminator()


discriminator.compile(optimizer=tf.keras.optimizers.Adam(0.0002, 0.5), loss="binary_crossentropy", metrics=["accuracy"])




In [12]:
discriminator.trainable = False  # Freeze the discriminator's weights when training the GAN
gan_input = layers.Input(shape=(input_dim,))
fake_data = generator(gan_input)
gan_output = discriminator(fake_data)
gan = tf.keras.Model(gan_input, gan_output)
gan.compile(optimizer=tf.keras.optimizers.Adam(0.0002, 0.5), loss="binary_crossentropy")


In [18]:
def train(data, epochs, batch_size):
    half_batch = int(batch_size / 2)


    for epoch in range(epochs):
        discriminator.trainable = True


        idx = np.random.randint(0, data.shape[0], half_batch)
        real_data = data.iloc[idx].values

        noise = np.random.normal(0, 1, (half_batch, input_dim))
        fake_data = generator.predict(noise)

        d_loss_real = discriminator.train_on_batch(real_data, np.ones((half_batch, 1)))
        d_loss_fake = discriminator.train_on_batch(fake_data, np.zeros((half_batch, 1)))
        d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

        discriminator.trainable = False


        noise = np.random.normal(0, 1, (batch_size, input_dim))

        g_loss = gan.train_on_batch(noise, np.ones((batch_size, 1)))


        if epoch % 100 == 0:
            print(f"epoch: {epoch+1}/{epochs} [D loss: {d_loss[0]:.4f}, acc.: {100*d_loss[1]:.2f}] [G loss: {g_loss}]")
        else:
            print(f"epoch: {epoch+1}/{epochs} ...")

In [19]:
train(data, epochs=100, batch_size=64)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
epoch: 1/100 [D loss: 0.2959, acc.: 95.50] [G loss: 0.9060183763504028]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
epoch: 2/100 ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
epoch: 3/100 ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
epoch: 4/100 ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
epoch: 5/100 ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
epoch: 6/100 ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
epoch: 7/100 ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
epoch: 8/100 ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
epoch: 9/100 ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
epoch: 10/100 ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
epoch: 11/100 ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
epoch: 12/100 ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
epoch: 13/100 ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
epoch: 14/100 ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
epoch: 15/100 ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
epoch: 16/100 ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
epoch: 17/100 ...
1/

In [21]:
def generate_synthetic_data(generator, num_samples=1):
    noise_dim=100
    # Generate random noise
    noise = np.random.normal(0, 1, (num_samples, noise_dim))

    # Generate synthetic data using the generator
    synthetic_data = generator.predict(noise)
    return synthetic_data


In [23]:
generate_synthetic_data(generator, num_samples=10)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step


array([[-4.38392222e-01,  2.16781050e-01,  6.85356976e-03,
         2.24885136e-01,  1.43757224e-01,  1.99965462e-01,
         9.91207242e-01, -1.38567463e-01,  9.98372078e-01,
         9.10532176e-02, -9.98106301e-01],
       [-1.84556544e-01,  1.70139283e-01, -1.87016383e-01,
         3.86069939e-02, -9.50110480e-02,  1.29389822e-01,
         9.60627437e-01,  1.38415515e-01,  9.90065634e-01,
        -5.27903810e-02, -9.90135193e-01],
       [-1.91733181e-01, -1.11060767e-02, -1.78532321e-02,
         3.32554318e-02,  7.34046996e-02,  1.94292039e-01,
         9.83040154e-01,  1.04949862e-01,  9.94276822e-01,
        -3.98635268e-02, -9.97956753e-01],
       [-1.16882108e-01,  1.00225978e-01, -1.03095022e-04,
         4.76264209e-02,  9.21659730e-03,  1.19465992e-01,
         9.76353884e-01,  1.49884087e-03,  9.93960023e-01,
         3.71492580e-02, -9.95670080e-01],
       [-7.73150753e-03, -1.01159802e-02, -8.87696743e-02,
         2.57639103e-02,  1.82826035e-02, -1.61726363e-02,
  